# Exercise 5 · Keep it on disk: hand the work to a new session

You will stop Codex halfway through a task, save what it knows in a file, and let a new session finish from that file.

**Goal:** Have Codex write `HANDOFF.md` after part 1 of a task. Then a new Codex session does part 2 from that file alone.

**Why it matters:** A Codex session holds only so much text: its context window. A new session remembers nothing. What must survive goes in a file.

**Before you start:** You can start Codex and leave it with `/exit` (exercise 1 of today).

**What the check measures:** `HANDOFF.md` has three headings (done, next, files that matter), `report.txt` lists the rooms busiest first, and your answer in Step 6 is right.

**Done when:** the last cell prints `Done:`.

## The situation

A team books three meeting rooms. `TASK.md` asks for a booking count in two parts. You do part 1 in one Codex session and part 2 in a new one. The files are made up.

## Step 1 · Write the practice files

This is the solved copy. Run the cell. It writes `practice/` as it is after Step 5: the Step 1 files, a `Start` commit, then `count.py`, `report.txt` and `HANDOFF.md` from Codex. It never overwrites a file.

Expect five `Wrote practice/` lines and `Ran count.py`.

In [ ]:
FILES = {
    "TASK.md": """# Task: count room bookings

Part 1. Write count.py. It reads bookings.csv and prints one line per room, for example `Room A: 3`.

Part 2. Change count.py so it also writes the same lines to report.txt, busiest room first.
""",
    "bookings.csv": """date,room,booked_by
2026-11-09,Room A,Team 1
2026-11-09,Room B,Team 2
2026-11-10,Room B,Team 3
2026-11-10,Room C,Team 1
2026-11-11,Room A,Team 2
2026-11-11,Room B,Team 1
2026-11-12,Room B,Team 2
2026-11-12,Room C,Team 3
2026-11-13,Room A,Team 3
2026-11-13,Room B,Team 1
""",
    ".gitignore": "__pycache__/\n",
}

# What Codex may leave after Steps 3 to 5 (simulated): count.py with both parts, and HANDOFF.md.
DONE = {
    "count.py": """\"\"\"Count bookings per room in bookings.csv; write report.txt, busiest room first.\"\"\"
import csv
from collections import Counter

with open("bookings.csv", newline="", encoding="utf-8") as sheet:
    counts = Counter(row["room"] for row in csv.DictReader(sheet))
for room in sorted(counts):
    print(f"{room}: {counts[room]}")
with open("report.txt", "w", encoding="utf-8") as report:
    for room, n in counts.most_common():
        report.write(f"{room}: {n}\\n")
""",
    "HANDOFF.md": """## Done
- Part 1: count.py prints bookings per room.
## Next
- Part 2 of TASK.md: also write the lines to report.txt, busiest room first.
## Files that matter
- TASK.md, bookings.csv, count.py
""",
}
import subprocess
import sys
from pathlib import Path

PRACTICE = Path("practice")
PRACTICE.mkdir(exist_ok=True)
started = (PRACTICE / ".git").exists()
for name, text in {**FILES, **DONE}.items():
    if (PRACTICE / name).exists():
        print(f"Kept practice/{name}: it is already there.")
    else:
        (PRACTICE / name).write_text(text, encoding="utf-8")
        print(f"Wrote practice/{name}")
if not started:
    subprocess.run(["git", "-C", str(PRACTICE), "init", "-q", "-b", "main"], check=True)
    subprocess.run(["git", "-C", str(PRACTICE), "add", *FILES], check=True)
    subprocess.run(["git", "-C", str(PRACTICE), "commit", "-qm", "Start"], check=True)
subprocess.run([sys.executable, "count.py"], cwd=PRACTICE, check=True, capture_output=True)
print("Ran count.py: it wrote practice/report.txt")

## Step 2 · Open a terminal and commit the start

Choose **Terminal > New Terminal**. Type each line, then press Enter:

```sh
source .venv/bin/activate
cd week-02/day-01/exercise-05-handoff-to-disk/practice
git init -q -b main && git add -A && git commit -qm "Start"
```

Expect no output from the last line.

### Learn: every skill solves a problem

The apprentice workspace has a skill called `handoff`. Its description says: "Compact the current conversation into a handoff document for another agent to pick up." ([handoff skill](https://github.com/GitRollTraining/fellowship-apprentice-workspace/blob/main/library/skills/handoff/SKILL.md)). It saves to your computer's temporary folder, so here you name the file yourself: `HANDOFF.md`.

## Step 3 · Part 1 with Codex

```sh
codex --no-daemon
```

If your Codex does not know `--no-daemon`, leave it out. If Codex asks "Trust this folder?", choose `1. Trust and continue`. Then send:

```text
Read TASK.md. Do Part 1 only, then stop. Run python count.py to test it.
```

Expect `count.py` and three lines: `Room A: 3`, `Room B: 5`, `Room C: 2`.

## Step 4 · Save what the next session needs

```text
Write HANDOFF.md for a new session that knows nothing. Use three headings: ## Done, ## Next, ## Files that matter. A few lines under each.
```

Type `/exit`. Then read the file:

```sh
cat HANDOFF.md
```

Expect the three headings. Ask yourself: could a stranger do part 2 from this file alone?

<details><summary>Show the answer</summary>

```markdown
## Done
- Part 1: count.py prints bookings per room.
## Next
- Part 2 of TASK.md: also write the lines to report.txt, busiest room first.
## Files that matter
- TASK.md, bookings.csv, count.py
```

Why: it names the next step and the files, so a new session needs nothing else.

</details>

## Step 5 · Part 2 in a new session

Start Codex again. Do not use `codex resume`: you want an empty chat.

```sh
codex --no-daemon
```

```text
Read HANDOFF.md and do what it lists under Next.
```

Type `/exit`, then run `cat report.txt`. Expect `Room B: 5`, `Room A: 3`, `Room C: 2`, in that order.

## Step 6 · Which limit does the skill solve?

Set `ANSWER` to one letter and run the cell.

- A. Codex cannot run Python scripts.
- B. A session's context window fills up, and a new session remembers nothing.
- C. Git cannot store Markdown files.
- D. Codex cannot read CSV files.

<details><summary>Show the answer</summary>

B. Why: the skill writes the conversation down so that "another agent" can pick it up. Codex's `/compact` is for the same limit: "Use after long runs so Codex retains key points without blowing the context window." ([CLI reference](https://learn.chatgpt.com/docs/developer-commands?surface=cli)).

</details>

In [ ]:
# Your answer: one letter, A, B, C or D
ANSWER = "B"
print(f"Your answer: {ANSWER or 'none yet'}")

## Start over (only if you need it)

Set `START_OVER = True` and run the cell. It removes `count.py`, `report.txt` and `HANDOFF.md` and writes the Step 1 files again. Then set it back to `False` and go to Step 3.

Expect `Nothing changed.` while it is `False`.

In [ ]:
# Start over: only if START_OVER is True, remove Codex's files and write the Step 1 files again
START_OVER = False

if not START_OVER:
    print("Nothing changed.")
elif "FILES" not in globals():
    print("Run the Step 1 cell first, then this cell.")
else:
    PRACTICE.mkdir(exist_ok=True)
    for name in ("count.py", "report.txt", "HANDOFF.md"):
        (PRACTICE / name).unlink(missing_ok=True)
    for name, text in FILES.items():
        (PRACTICE / name).write_text(text, encoding="utf-8")
    print("Wrote the practice files again. Now set START_OVER back to False.")

## Check your work

Run the cell. Expect three `ok:` lines, then `Done:`. A `Not yet:` line names the first problem and what to do.

In [ ]:
# Check your work: read HANDOFF.md, report.txt and your answer
from pathlib import Path

PRACTICE = Path("practice")
HEADINGS = ("done", "next", "files that matter")
REPORT = ["Room B: 5", "Room A: 3", "Room C: 2"]


def first_problem() -> str:
    """Print ok for each check that passes; return the first problem, or ""."""
    handoff = PRACTICE / "HANDOFF.md"
    if not handoff.is_file():
        return "practice/HANDOFF.md is missing. Ask Codex to write it (Step 4)."
    titles = [line.lstrip("#").strip().lower() for line in handoff.read_text(encoding="utf-8").splitlines()
              if line.startswith("#")]
    missing = [h for h in HEADINGS if not any(h in t for t in titles)]
    if missing:
        return f"HANDOFF.md has no heading for {missing[0]!r}. Ask Codex to add it (Step 4)."
    print("ok: HANDOFF.md has the three headings")
    report = PRACTICE / "report.txt"
    if not report.is_file():
        return "practice/report.txt is missing. Do part 2 in a new session (Step 5)."
    lines = [line.strip() for line in report.read_text(encoding="utf-8").splitlines() if line.strip()]
    if lines != REPORT:
        return f"report.txt holds {lines}, not {REPORT}. Ask Codex to fix part 2 (Step 5)."
    print("ok: report.txt lists the rooms, busiest first")
    answer = str(globals().get("ANSWER", "")).strip().upper()
    if not answer:
        return "no answer yet. Set ANSWER and run its cell (Step 6)."
    if answer != "B":
        return f"answer {answer} is not the limit the skill solves. Read Step 6 again."
    print("ok: you named the limit the skill solves")
    return ""


problem = first_problem()
print(f"Not yet: {problem}" if problem else
      "Done: a new session finished the task from a file on disk.")

## What you learned

- A session forgets. A file on disk does not.
- A good handoff names what is done, what is next and which files matter.

**Reflect:** Which of your own tasks would you split across two sessions? What would its `HANDOFF.md` say?